# AI CEO Agent

This is Task 5 + Task 6 together: it reads the outputs of the opportunity,
risk, trend, and competitor agents, and turns them into prioritized,
evidence-backed strategic recommendations - plus a short CEO briefing.

Important: this agent does **not** go back to the raw articles. It reasons
over the already-extracted intelligence (the 4 JSON files), the same way a
CEO would read analyst summaries rather than every news article personally.


In [ ]:
import json
import os
import time

import requests
from pathlib import Path

cwd = Path.cwd()
DATA_DIR = next((p for p in [cwd / "notebook" / "data", cwd.parent / "notebook" / "data"] if p.exists()), cwd)

OPPORTUNITIES_PATH = DATA_DIR / "opportunities.json"
RISKS_PATH         = DATA_DIR / "risks.json"
TRENDS_PATH        = DATA_DIR / "trends.json"
COMPETITORS_PATH   = DATA_DIR / "competitor_activity.json"

RECOMMENDATIONS_OUT = DATA_DIR / "recommendations.json"
BRIEFING_OUT        = DATA_DIR / "ceo_briefing.json"

OLLAMA_HOST  = os.getenv("OLLAMA_HOST", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "phi4-mini")
TIMEOUT      = 300

print("data dir:", DATA_DIR)


In [ ]:
with open(OPPORTUNITIES_PATH) as f:
    opportunities = json.load(f)
with open(RISKS_PATH) as f:
    risks = json.load(f)
with open(TRENDS_PATH) as f:
    trends = json.load(f)
with open(COMPETITORS_PATH) as f:
    competitors = json.load(f)

print("opportunities:", len(opportunities))
print("risks:", len(risks))
print("trends:", len(trends))
print("competitors:", len(competitors))


In [ ]:
def ask_llm(prompt):
    resp = requests.post(
        f"{OLLAMA_HOST}/api/generate",
        json={"model": OLLAMA_MODEL, "prompt": prompt, "stream": False},
        timeout=TIMEOUT,
    )
    resp.raise_for_status()
    return resp.json()["response"]


def warmup():
    print("warming up model...")
    start = time.time()
    ask_llm("hi")
    print("warmup done in", round(time.time() - start, 1), "sec")


warmup()


## Build a summary of all the intelligence

Turns the 4 JSON files into a short bullet list the CEO agent can reason
over - this is the "analyst briefing" the CEO agent reads.


In [ ]:
def summarize_items(items, label):
    lines = [f"{label}:"]
    for item in items:
        title = item.get("title") or item.get("competitor", "Unknown")
        extra = item.get("evidence") or item.get("summary") or ""
        lines.append(f"- {title}: {extra}")
    return "\n".join(lines)


briefing_context = "\n\n".join([
    summarize_items(opportunities, "OPPORTUNITIES"),
    summarize_items(risks, "RISKS"),
    summarize_items(trends, "TRENDS"),
    summarize_items(competitors, "COMPETITOR ACTIVITY"),
])

print(briefing_context[:1000])


## CEO agent - strategic recommendations

For each recommendation: what to do, how urgent it is, what evidence backs
it up, what the expected upside is, and what could go wrong. This matches
both Task 6 (evidence-based recommendations) and the dashboard's Strategic
Recommendations section.


In [ ]:
def ceo_agent(context):
    prompt = f"""You are the AI CEO advisor for SAP. Below is a summary of
opportunities, risks, trends, and competitor activity gathered from recent news.

Your job is NOT to just repeat the information. Reason about what it means for
the business and recommend 3 to 5 concrete strategic actions, ranked by priority.

Return ONLY a JSON array, no extra text, no markdown. Each item must look like this:

{{
  "recommendation": "a clear, specific action to take",
  "priority": "High, Medium, or Low",
  "supporting_evidence": ["short evidence point 1", "short evidence point 2"],
  "expected_impact": "what improves if this is done (e.g. revenue growth, market differentiation)",
  "risk_level": "High, Medium, or Low"
}}

Intelligence summary:
{context}
"""
    raw = ask_llm(prompt)
    return raw


def parse_json_list(raw_text):
    text = raw_text.strip()
    if text.startswith("```"):
        text = text.strip("`")
        text = text.replace("json", "", 1).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        print("could not parse JSON, here is the raw text:")
        print(text)
        return []


raw_output = ceo_agent(briefing_context)
recommendations = parse_json_list(raw_output)

print("parsed", len(recommendations), "recommendations")
for r in recommendations:
    print("-", r.get("priority"), "|", r.get("recommendation"))


In [ ]:
with open(RECOMMENDATIONS_OUT, "w") as f:
    json.dump(recommendations, f, indent=2)

print("saved to", RECOMMENDATIONS_OUT)


## CEO briefing - the executive summary

Three questions the dashboard's CEO Briefing section needs answered:
what happened, why it matters, and what management should do next.


In [ ]:
def ceo_briefing_agent(context, recommendations):
    rec_text = "\n".join(f"- {r.get('recommendation')}" for r in recommendations)

    prompt = f"""You are writing a short executive briefing for the CEO of SAP.

Use the intelligence summary and recommendations below to answer exactly
three questions in plain, direct language. No fluff, no buzzwords.

Return ONLY a JSON object, no extra text, no markdown, in this format:

{{
  "what_happened": "2-3 sentences on the key developments",
  "why_it_matters": "2-3 sentences on the business implications",
  "what_to_do_next": "2-3 sentences summarizing the top priorities"
}}

Intelligence summary:
{context}

Recommendations already identified:
{rec_text}
"""
    raw = ask_llm(prompt)
    return raw


def parse_json_obj(raw_text):
    text = raw_text.strip()
    if text.startswith("```"):
        text = text.strip("`")
        text = text.replace("json", "", 1).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        print("could not parse JSON, here is the raw text:")
        print(text)
        return None


raw_briefing = ceo_briefing_agent(briefing_context, recommendations)
ceo_briefing = parse_json_obj(raw_briefing)

print(json.dumps(ceo_briefing, indent=2))


## Save the briefing for the dashboard

In [ ]:
with open(BRIEFING_OUT, "w") as f:
    json.dump(ceo_briefing, f, indent=2)

print("saved to", BRIEFING_OUT)
